<a href="https://colab.research.google.com/github/leo-aguiar/ibm_hr_analytics_employee/blob/main/IBM_Analytics_Employee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bibliotecas para manipulação, visualização e ingestão de dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub as kagg
import os

In [ ]:
# Realiza a ingestão do dataset diretamente do Kaggle para o ambiente local
path = kagg.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")

In [ ]:
# Carrega o dataset de rotatividade de funcionários em um DataFrame para posterior análise exploratória e modelagem
df = pd.read_csv(os.path.join(path, "WA_Fn-UseC_-HR-Employee-Attrition.csv"))

In [ ]:
# Seleciona amostra aleatória de registros para inspeção inicial dos dados
df.sample(5)

In [ ]:
# Inspeciona a estrutura geral do dataset para avaliar volume de registros, tipos de dados, completude das colunas e possíveis necessidades de pré-processamento
df.info()

In [ ]:
# Avalia a completude do dataset ao consolidar, por coluna,
# a quantidade absoluta e percentual de valores ausentes por vaariável
missing_table = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().mean() * 100).round(2)
})
missing_table = missing_table.sort_values(by='Missing %', ascending=False)
missing_table

In [ ]:
# Annotation: Não foram identificados valores ausentes no dataset, indicando completude total das variáveis e eliminando a necessidade de tartamento de missinbg values nesta etapa do pré-processamento.

In [ ]:
# Segmenta as variáveis conforme seu tipo de dado para direcionar análises específicas de atributos numéricos e categóricos
numerical_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

In [ ]:
numerical_cols

In [ ]:
categorical_cols

In [ ]:
# Variáveis categóricas ordinais codificadas numericamente
ordinal_cols = [
    'Education',
    'EnvironmentSatisfaction',
    'JobInvolvement',
    'JobSatisfaction',
    'PerformanceRating',
    'RelationshipSatisfaction',
    'WorkLifeBalance'
]

In [ ]:
# Exclui features ordinais da lista numérica por representarem categorias ordenadas, não valores contínuos
numerical_cols = [
    col for col in numerical_cols
    if col not in ordinal_cols
]

In [ ]:
numerical_cols

In [ ]:
# Analisa a distribuição da variável alvo para compreender a taxa de attrition e avaliar o balanceamento entre as classes do problema
df['Attrition'].value_counts()

In [ ]:
# Analisa a distribuição da variável alvo para compreender a taxa de attrition e avaliar o balanceamento entre as classes do problema
target_table = pd.DataFrame({
    'Contagem': df['Attrition'].value_counts(),
    'Taxa (%)': (df['Attrition'].value_counts(normalize=True) * 100).round(2)
})
target_table

In [ ]:
# Annotation: A variável alvo apresenta desbalanceamento moderado, com predominância da classe negativa, o que deverá ser considerado nas etapas de modelagem e avaliação de performance

In [ ]:
# Gera estatísticas descritivas das variáveis numéricas para avaliar distribuição, tendência central, dispersão e possíveis valores extremos
numerical_summary = df[numerical_cols].describe().T
numerical_summary

In [ ]:
# Annotation: As colunas EmployeeCount e EmployeeCount não carregam informação útil para análise/modelagem, portanto podem ser removidas

In [ ]:
# Visualiza a distribuição das variáveis numéricas para identificar padrões, assimetrias e possíveis valores extremos
df[numerical_cols].hist(figsize=(12, 10))
plt.tight_layout()
plt.show()

In [ ]:
# Analisa a dispersão das variáveis numéricas para identificar possíveis outliers e padrões de distribuição dos dados
plt.figure(figsize=(12, 10))

for i, col in enumerate(numerical_cols, 1):
    plt.subplot(5, 4, i)
    sns.boxplot(x=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
# Analisa a relação entre realização de horas extras e taxa de attrition para investigar possível impacto da carga de trabalho na rotatividade
overtime_attrition = pd.crosstab(
    df['OverTime'],
    df['Attrition'],
    normalize='index'
) * 100

overtime_attrition

In [ ]:
# Funcionários que realizam horas extras apresentam taxa de attrition significativamente superior aos que não realizam overtime (3x mais aproximadamente), sugerindo possível impacto da sobrecarga de trabalho na retenção.

In [ ]:
# Analisa a relação entre nível de satisfação no trabalho e taxa de attrition para investigar possível impacto da satisfação na retenção de funcionários
jobsat_attrition = pd.crosstab(
    df['JobSatisfaction'],
    df['Attrition'],
    normalize='index'
) * 100

jobsat_attrition

In [ ]:
# Observa-se uma relação inversa entre satisfação no trabalho e attrition, indicando que funcionários menos satisfeitos apresentam maiores taxas de saída da empresa

# A relação não é perfeitamente linear

In [ ]:
# Analisa a relação entre equilíbrio vida-trabalho e taxa de attrition para investigar possível impacto do bem-estar na retenção de funcionários

worklife_attrition = pd.crosstab(
    df['WorkLifeBalance'],
    df['Attrition'],
    normalize='index'
) * 100

worklife_attrition

In [ ]:
# Funcionários com pior equilíbrio entre vida pessoal e trabalho (nível 1) apresentam taxa de attrition significativamente superior aos demais grupos. Entretanto, não foi observada uma relação linear entre os níveis subsequentes da variável, sugerindo que o principal fator de risco está associado a percepções muito negativas de equilíbrio vida-trabalho.

In [ ]:
# Analisa a quantidade de funcionários em cada nível de WorkLifeBalance
df['WorkLifeBalance'].value_counts().sort_index()

In [ ]:
# Há evidências de que funcionários com pior equilíbrio entre vida pessoal e trabalho apresentam maior propensão à saída da empresa.

In [ ]:
# Compara a distribuição salarial entre funcionários que permaneceram e os que deixaram a empresa para investigar possível relação entre renda e attrition
plt.figure(figsize=(8, 5))

sns.boxplot(
    x='Attrition',
    y='MonthlyIncome',
    data=df
)

plt.title('Monthly Income vs Attrition')
plt.show()

In [ ]:
# Os funcionários que permaneceram na empresa apresentam renda mensal superior aos que deixaram a organização. A mediana salarial do grupo sem attrition é significativamente maior, sugerindo uma associação entre remuneração e retenção de talentos.

In [ ]:
# Compara a distribuição de idade entre funcionários que permaneceram e os que deixaram a empresa para investigar possível relação entre faixa etária e attrition
plt.figure(figsize=(8, 5))

sns.boxplot(
    x='Attrition',
    y='Age',
    data=df
)

plt.title('Age vs Attrition')
plt.show()

In [ ]:
# Funcionários que deixaram a empresa apresentam idade inferior aos que permaneceram. A distribuição etária do grupo com attrition encontra-se deslocada para faixas mais jovens, sugerindo associação entre menor idade e maior propensão à saída da organização.

In [ ]:
# Verifica a distribuição da variável Over18 para avaliar sua variabilidade e relevância analítica
df['Over18'].value_counts()